# Week 4 Exercise - Python to JavaScript Code Converter

A Gradio-powered tool that converts Python code to optimized JavaScript using frontier models.

## Features
- Convert Python to JavaScript with AI models
- Execute both Python and JavaScript code side-by-side
- Multiple model support via OpenRouter
- Performance comparison between implementations

## Why JavaScript?
- Universal runtime (browser + Node.js)
- No compilation step needed
- Easy to test and deploy
- Useful for web development and serverless functions

In [ ]:
# Imports

import os
import io
import sys
import json
import shutil
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Environment Setup

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"API Key exists and begins {openai_api_key[:8]}")
else:
    print("API Key not set")

# Check for Node.js
node_path = shutil.which("node")
if node_path:
    try:
        version = subprocess.check_output(["node", "--version"], text=True).strip()
        print(f"Node.js installed: {version}")
    except:
        print("Node.js found but version check failed")
else:
    print("Node.js not installed - JavaScript execution will be disabled")

In [ ]:
# Setup OpenRouter Client

# Works with OpenRouter (sk-or-*) or direct OpenAI keys
if openai_api_key and openai_api_key.startswith('sk-or-'):
    client = OpenAI(
        api_key=openai_api_key,
        base_url="https://openrouter.ai/api/v1"
    )
    print("Using OpenRouter")
else:
    client = OpenAI()
    print("Using OpenAI directly")

# Available models via OpenRouter
MODELS = [
    "gpt-4.1-mini",
    "gpt-4.1",
    "anthropic/claude-sonnet-4",
    "google/gemini-2.0-flash-exp:free",
    "deepseek/deepseek-coder",
    "qwen/qwen-2.5-coder-32b-instruct",
]

print(f"Available models: {len(MODELS)}")

In [ ]:
# System Information

import platform

def get_system_info():
    """Gather system information for the LLM context."""
    info = {
        "os": platform.system(),
        "os_version": platform.version(),
        "architecture": platform.machine(),
        "python_version": platform.python_version(),
        "node_installed": bool(shutil.which("node")),
    }
    
    if info["node_installed"]:
        try:
            info["node_version"] = subprocess.check_output(
                ["node", "--version"], text=True
            ).strip()
        except:
            info["node_version"] = "unknown"
    
    return info

system_info = get_system_info()
print(json.dumps(system_info, indent=2))

In [ ]:
# Prompts for Code Conversion

SYSTEM_PROMPT = """
Your task is to convert Python code into high-performance JavaScript code.
Respond only with JavaScript code. Do not provide any explanation other than occasional comments.

Requirements:
1. The JavaScript code must produce IDENTICAL output to the Python code
2. Optimize for fastest possible execution
3. Use modern JavaScript (ES6+) features
4. The code should run in Node.js
5. Use console.log() for output (equivalent to print())
6. Handle Python-specific constructs appropriately:
   - range() -> for loops or Array.from()
   - list comprehensions -> map/filter or loops
   - f-strings -> template literals
   - time.time() -> performance.now() or Date.now()

Important:
- Ensure numeric precision matches Python output
- Handle large numbers correctly (use BigInt if needed)
- Output should be exactly the same as Python
"""

def create_user_prompt(python_code: str) -> str:
    """Create the user prompt for code conversion."""
    return f"""
Convert this Python code to JavaScript that produces identical output.

System information:
{json.dumps(system_info, indent=2)}

The JavaScript will be saved to main.js and run with: node main.js

Respond only with JavaScript code. No markdown fences or explanation.

Python code:

```python
{python_code}
```
"""

In [ ]:
# Code Conversion Function

def convert_to_javascript(model: str, python_code: str) -> str:
    """Convert Python code to JavaScript using the selected model."""
    
    if not python_code.strip():
        return "// No Python code provided"
    
    try:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": create_user_prompt(python_code)}
        ]
        
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=4096,
            temperature=0.2
        )
        
        js_code = response.choices[0].message.content or ""
        
        # Clean up markdown fences if present
        for fence in ["```javascript", "```js", "```"]:
            js_code = js_code.replace(fence, "")
        
        return js_code.strip()
        
    except Exception as e:
        return f"// Error: {str(e)}"

In [ ]:
# Code Execution Functions

def run_python(code: str) -> str:
    """Execute Python code and capture output."""
    if not code.strip():
        return "No code to execute"
    
    globals_dict = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    
    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout
    
    return output if output else "(no output)"


def run_javascript(code: str) -> str:
    """Execute JavaScript code using Node.js."""
    if not code.strip():
        return "No code to execute"
    
    if not shutil.which("node"):
        return "Error: Node.js is not installed. Install it from https://nodejs.org/"
    
    # Write code to file
    try:
        with open("main.js", "w") as f:
            f.write(code)
        
        # Run with Node.js
        result = subprocess.run(
            ["node", "main.js"],
            capture_output=True,
            text=True,
            timeout=30
        )
        
        if result.returncode != 0:
            return f"Error:\n{result.stderr}"
        
        return result.stdout if result.stdout else "(no output)"
        
    except subprocess.TimeoutExpired:
        return "Error: Execution timed out (30s limit)"
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
# Example Python Code

EXAMPLE_PI = '''
import time

def calculate_pi(iterations):
    """Calculate pi using Leibniz formula."""
    result = 0.0
    for i in range(iterations):
        sign = 1 if i % 2 == 0 else -1
        result += sign / (2 * i + 1)
    return result * 4

start = time.time()
pi = calculate_pi(1_000_000)
end = time.time()

print(f"Pi approximation: {pi:.10f}")
print(f"Execution time: {(end - start):.6f} seconds")
'''

EXAMPLE_FIBONACCI = '''
import time

def fibonacci(n):
    """Calculate nth Fibonacci number iteratively."""
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

start = time.time()
result = fibonacci(40)
end = time.time()

print(f"Fibonacci(40) = {result}")
print(f"Execution time: {(end - start):.6f} seconds")
'''

EXAMPLE_PRIMES = '''
import time

def sieve_of_eratosthenes(limit):
    """Find all primes up to limit using Sieve of Eratosthenes."""
    is_prime = [True] * (limit + 1)
    is_prime[0] = is_prime[1] = False
    
    for i in range(2, int(limit ** 0.5) + 1):
        if is_prime[i]:
            for j in range(i * i, limit + 1, i):
                is_prime[j] = False
    
    return sum(is_prime)

start = time.time()
count = sieve_of_eratosthenes(1_000_000)
end = time.time()

print(f"Primes up to 1,000,000: {count}")
print(f"Execution time: {(end - start):.6f} seconds")
'''

EXAMPLES = {
    "Pi Calculation (Leibniz)": EXAMPLE_PI,
    "Fibonacci (Iterative)": EXAMPLE_FIBONACCI,
    "Prime Sieve": EXAMPLE_PRIMES
}

print(f"Example algorithms: {list(EXAMPLES.keys())}")

In [ ]:
# Gradio UI

def load_example(example_name: str) -> str:
    """Load an example Python code snippet."""
    return EXAMPLES.get(example_name, "")


with gr.Blocks(title="Python to JavaScript Converter", theme=gr.themes.Soft()) as app:
    gr.Markdown("""
    # Python to JavaScript Code Converter
    
    Convert Python code to optimized JavaScript using AI models.
    Execute both versions to compare output and performance.
    """)
    
    with gr.Row():
        example_dropdown = gr.Dropdown(
            choices=list(EXAMPLES.keys()),
            label="Load Example",
            value=list(EXAMPLES.keys())[0]
        )
        model_dropdown = gr.Dropdown(
            choices=MODELS,
            value=MODELS[0],
            label="Model"
        )
    
    with gr.Row(equal_height=True):
        with gr.Column():
            python_code = gr.Code(
                label="Python Code",
                language="python",
                lines=20,
                value=EXAMPLE_PI.strip()
            )
        with gr.Column():
            js_code = gr.Code(
                label="JavaScript Code",
                language="javascript",
                lines=20,
                value="// Click 'Convert to JavaScript' to generate code"
            )
    
    with gr.Row():
        run_python_btn = gr.Button("Run Python", variant="secondary")
        convert_btn = gr.Button("Convert to JavaScript", variant="primary")
        run_js_btn = gr.Button("Run JavaScript", variant="secondary")
    
    with gr.Row(equal_height=True):
        with gr.Column():
            python_output = gr.Textbox(
                label="Python Output",
                lines=6,
                interactive=False
            )
        with gr.Column():
            js_output = gr.Textbox(
                label="JavaScript Output",
                lines=6,
                interactive=False
            )
    
    gr.Markdown("""
    ### Notes
    - JavaScript execution requires Node.js to be installed
    - Output should be identical between Python and JavaScript
    - Performance may vary based on the algorithm and implementation
    """)
    
    # Event handlers
    example_dropdown.change(
        fn=load_example,
        inputs=[example_dropdown],
        outputs=[python_code]
    )
    
    convert_btn.click(
        fn=convert_to_javascript,
        inputs=[model_dropdown, python_code],
        outputs=[js_code]
    )
    
    run_python_btn.click(
        fn=run_python,
        inputs=[python_code],
        outputs=[python_output]
    )
    
    run_js_btn.click(
        fn=run_javascript,
        inputs=[js_code],
        outputs=[js_output]
    )

In [ ]:
# Launch the app
app.launch()

## Usage Guide

### How to Use
1. **Load Example** or write your own Python code
2. **Select Model** - Choose which AI model to use for conversion
3. **Run Python** - Execute the Python code to see expected output
4. **Convert to JavaScript** - Generate JavaScript equivalent
5. **Run JavaScript** - Execute and compare output

### Requirements
- Python 3.8+
- Node.js (for JavaScript execution)
- OpenRouter API key (or OpenAI API key)

### Supported Python Constructs
- Basic arithmetic and logic
- Functions and loops
- List comprehensions
- f-strings (converted to template literals)
- time module (converted to performance.now())

### Performance Notes
- JavaScript may be faster or slower depending on the algorithm
- Node.js JIT compilation can provide significant speedups
- Large number handling may differ between languages